In [ ]:
# Cell 1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plot style
plt.style.use('seaborn-v0_8')
sns.set_palette("Set2")

In [ ]:
# Cell 2
# Load the cleaned dataset with timestamps
df = pd.read_csv('../data/cleaned_ratings_with_time.csv')

# Convert timestamp_datetime string back to datetime if needed
df['timestamp_datetime'] = pd.to_datetime(df['timestamp_datetime'])

# Display basic info
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Cell 3
# Check data types
print(df.dtypes)

# Check for missing values
print(f"\nMissing values per column:\n{df.isnull().sum()}")

# Summary statistics for numeric columns
df.describe()

In [ ]:
# Cell 4
# Rating distribution
rating_counts = df['rating'].value_counts().sort_index()
print(f"Rating counts:\n{rating_counts}")
print(f"Percentage distribution:\n{rating_counts/len(df)*100}")

# Plot histogram
fig, ax = plt.subplots(figsize=(8,5))
df['rating'].hist(bins=5, edgecolor='black', alpha=0.7, ax=ax)
ax.set_title('Distribution of Ratings', fontsize=14)
ax.set_xlabel('Rating')
ax.set_ylabel('Frequency')
ax.set_xticks([1,2,3,4,5])
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5
# Ratings per user
ratings_per_user = df.groupby('user_id').size()

print(f"Users: {len(ratings_per_user)}")
print(f"Min ratings by a user: {ratings_per_user.min()}")
print(f"Max ratings by a user: {ratings_per_user.max()}")
print(f"Mean ratings per user: {ratings_per_user.mean():.1f}")
print(f"Median ratings per user: {ratings_per_user.median():.1f}")

# Histogram with log scale
fig, ax = plt.subplots(figsize=(8,5))
ratings_per_user.hist(bins=50, edgecolor='black', alpha=0.7, ax=ax)
ax.set_title('Distribution of Ratings per User', fontsize=14)
ax.set_xlabel('Number of ratings')
ax.set_ylabel('Number of users')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6
# Percentage of users with few ratings
cold_users = (ratings_per_user < 20).sum()
print(f"Users with <20 ratings: {cold_users} ({cold_users/len(ratings_per_user)*100:.1f}%)")

cold_users_very = (ratings_per_user < 5).sum()
print(f"Users with <5 ratings: {cold_users_very} ({cold_users_very/len(ratings_per_user)*100:.1f}%)")

In [ ]:
# Cell 7
# Ratings per movie
ratings_per_movie = df.groupby('item_id').size()

print(f"Movies: {len(ratings_per_movie)}")
print(f"Min ratings for a movie: {ratings_per_movie.min()}")
print(f"Max ratings for a movie: {ratings_per_movie.max()}")
print(f"Mean ratings per movie: {ratings_per_movie.mean():.1f}")
print(f"Median ratings per movie: {ratings_per_movie.median():.1f}")

# Top 10 most rated movies
top_movies = df.groupby('title').size().sort_values(ascending=False).head(10)
print(f"\nTop 10 most rated movies:\n{top_movies}")

# Movies with only 1 rating
single_rating_movies = (ratings_per_movie == 1).sum()
print(f"\nMovies with only 1 rating: {single_rating_movies} ({single_rating_movies/len(ratings_per_movie)*100:.1f}%)")

In [ ]:
# Cell 8
# Calculate sparsity
n_users = df['user_id'].nunique()
n_movies = df['item_id'].nunique()
n_ratings = len(df)
total_possible = n_users * n_movies
sparsity = 1 - (n_ratings / total_possible)

print(f"Number of users: {n_users}")
print(f"Number of movies: {n_movies}")
print(f"Number of ratings: {n_ratings}")
print(f"Total possible ratings: {total_possible:,}")
print(f"Sparsity: {sparsity:.2%}")
print(f"Filled percentage: {(1-sparsity):.2%}")

In [ ]:
# Cell 9
# Extract time features
df['year'] = df['timestamp_datetime'].dt.year
df['month'] = df['timestamp_datetime'].dt.month
df['day_of_week'] = df['timestamp_datetime'].dt.dayofweek
df['hour'] = df['timestamp_datetime'].dt.hour

# Ratings per month
ratings_by_month = df.set_index('timestamp_datetime').resample('M').size()

fig, ax = plt.subplots(figsize=(12,5))
ratings_by_month.plot(ax=ax)
ax.set_title('Number of Ratings Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Number of ratings')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10
# Average rating per month
avg_rating_by_month = df.set_index('timestamp_datetime').resample('M')['rating'].mean()

fig, ax = plt.subplots(figsize=(12,5))
avg_rating_by_month.plot(ax=ax)
ax.set_title('Average Rating Over Time', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Average rating')
ax.set_ylim(3, 4.5)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11
# Average rating by day of week (0=Monday, 6=Sunday)
rating_by_dow = df.groupby('day_of_week')['rating'].mean()
dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, ax = plt.subplots(figsize=(10,4))
rating_by_dow.plot(kind='bar', ax=ax)
ax.set_title('Average Rating by Day of Week', fontsize=14)
ax.set_xlabel('Day of week')
ax.set_ylabel('Average rating')
ax.set_xticklabels(dow_names, rotation=45)
ax.axhline(y=df['rating'].mean(), color='red', linestyle='--', label='Overall average')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12
# Average rating by hour
rating_by_hour = df.groupby('hour')['rating'].mean()
ratings_count_by_hour = df.groupby('hour').size()

fig, axes = plt.subplots(2, 1, figsize=(12,8))

# Average rating by hour
rating_by_hour.plot(kind='bar', ax=axes[0])
axes[0].set_title('Average Rating by Hour of Day', fontsize=14)
axes[0].set_xlabel('Hour (0-23)')
axes[0].set_ylabel('Average rating')
axes[0].axhline(y=df['rating'].mean(), color='red', linestyle='--')

# Number of ratings by hour
ratings_count_by_hour.plot(kind='bar', ax=axes[1])
axes[1].set_title('Number of Ratings by Hour of Day', fontsize=14)
axes[1].set_xlabel('Hour (0-23)')
axes[1].set_ylabel('Number of ratings')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 13
# Compare weekend (Saturday=5, Sunday=6) vs weekday
df['is_weekend'] = df['day_of_week'].isin([5, 6])

weekend_rating = df[df['is_weekend']]['rating'].mean()
weekday_rating = df[~df['is_weekend']]['rating'].mean()

print(f"Weekend average rating: {weekend_rating:.3f}")
print(f"Weekday average rating: {weekday_rating:.3f}")
print(f"Difference: {weekend_rating - weekday_rating:.3f}")

In [ ]:
# Cell 14
# Create summary dictionary
summary = {
    'Metric': [
        'Total ratings',
        'Unique users', 
        'Unique movies',
        'Sparsity (%)',
        'Most common rating',
        'Ratings per user (median)',
        'Ratings per movie (median)',
        'Time span (years)',
        'Weekend vs weekday diff'
    ],
    'Value': [
        f"{len(df):,}",
        n_users,
        n_movies,
        f"{sparsity:.1%}",
        df['rating'].mode()[0],
        f"{ratings_per_user.median():.1f}",
        f"{ratings_per_movie.median():.1f}",
        f"{(df['timestamp_datetime'].max() - df['timestamp_datetime'].min()).days / 365:.1f}",
        f"{weekend_rating - weekday_rating:.3f}"
    ]
}

summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
# Cell 15
# Save key figures for your technical report
import os
os.makedirs('../figures', exist_ok=True)

# Recreate and save rating distribution
fig, ax = plt.subplots(figsize=(8,5))
df['rating'].hist(bins=5, edgecolor='black', alpha=0.7)
ax.set_title('Distribution of Ratings')
ax.set_xlabel('Rating')
ax.set_ylabel('Frequency')
plt.savefig('../figures/rating_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

print("Figures saved to ../figures/")